# DustiniaDelixia Groceria - Operational Analysis EDA
This notebook performs Exploratory Data Analysis (EDA) on the DustiniaDelixia Groceria dataset to identify delivery bottlenecks, geographic distributions, and data anomalies. All processing is done strictly using PySpark DataFrames, with Pandas only used for final rendering in visualizations.

In [ ]:
%pip install -r /home/jovyan/jupyter_requirements.txt
import os
import zipfile
import gdown
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, datediff, when, count, isnan, isnull, avg, hour, concat_ws, lit

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DustiniaDelixia_Operational_EDA") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

data_path = "./raw_data"


## 1. Load Data into Spark DataFrames
We will load the relevant tables for the Operational Analyst persona. Zero pandas loading used here.


In [ ]:
orders = spark.read.csv(f"{data_path}/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{data_path}/order_items.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{data_path}/sellers.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{data_path}/customers.csv", header=True, inferSchema=True)
reviews = spark.read.csv(f"{data_path}/order_reviews.csv", header=True, inferSchema=True)


## 2. Analyze Order Timestamps & Delivery Times
Let's find the distribution of delivery times and time-of-day bottlenecks using Spark.


In [ ]:
# Convert timestamp columns to proper timestamp type
timestamp_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for c in timestamp_cols:
    orders = orders.withColumn(c, to_timestamp(col(c)))

# Calculate delivery time in days
orders = orders.withColumn("delivery_time_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
orders = orders.withColumn("is_delayed", (col("order_delivered_customer_date") > col("order_estimated_delivery_date")).cast("int"))
orders = orders.withColumn("purchase_hour", hour(col("order_purchase_timestamp")))

# Aggregate delay rate by purchase hour
hourly_delays = orders.filter(col("is_delayed").isNotNull()).groupBy("purchase_hour").agg(
    avg("is_delayed").alias("delay_rate"),
    count("order_id").alias("total_orders")
).orderBy("purchase_hour")

# Convert ONLY the aggregated result to pandas for plotting
hourly_delays_pd = hourly_delays.toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=hourly_delays_pd, x="purchase_hour", y="delay_rate", color="salmon")
plt.title("Delivery Delay Rate by Purchase Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Delay Rate")
plt.show()


## 3. Route Analysis & Freight Value Impact
Let's join order items, sellers, and customers to map routes and check freight value correlations using Spark.


In [ ]:
# Join dataframes using Spark
fact = orders.join(order_items, on="order_id", how="left") \
    .join(sellers, on="seller_id", how="left") \
    .join(customers, on="customer_id", how="left")

# Create Route Column
fact = fact.withColumn("route", concat_ws(" -> ", col("seller_state"), col("customer_state")))

# Aggregate by Route
route_stats = fact.filter(col("is_delayed").isNotNull()).groupBy("route").agg(
    avg("is_delayed").alias("delay_rate"),
    avg("freight_value").alias("avg_freight"),
    avg("delivery_time_days").alias("avg_delivery_days"),
    count("order_id").alias("order_count")
).filter(col("order_count") > 50).orderBy(col("delay_rate").desc())

# Convert to Pandas for visualization
route_stats_pd = route_stats.limit(10).toPandas()

plt.figure(figsize=(12, 6))
sns.barplot(data=route_stats_pd, y="route", x="delay_rate", hue="route", legend=False, palette="Reds_r")
plt.title("Top 10 Worst Performing Routes by Delay Rate")
plt.xlabel("Delay Rate")
plt.ylabel("Route (Seller State -> Customer State)")
plt.show()

# Freight value vs Delivery Days correlation
# We'll aggregate by freight value bins to keep the plot fast and distributed
fact_with_bins = fact.withColumn("freight_bin", (col("freight_value") / 10).cast("int") * 10)
freight_stats = fact_with_bins.groupBy("freight_bin").agg(
    avg("delivery_time_days").alias("avg_delivery_days"),
    count("order_id").alias("order_count")
).filter(col("order_count") > 100).orderBy("freight_bin")

freight_pd = freight_stats.toPandas()

plt.figure(figsize=(10, 6))
sns.regplot(data=freight_pd, x="freight_bin", y="avg_delivery_days", scatter_kws={'s': freight_pd['order_count'].astype(float)/10})
plt.title("Avg Delivery Days by Freight Value Bin")
plt.xlabel("Freight Value Bin")
plt.ylabel("Avg Delivery Days")
plt.show()
